In [ ]:
!pip install nilearn scikit-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title Importing Libraries
# Cell 1: Install and Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import BayesianRidge, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import pickle
import os

print("✅ Libraries imported")

✅ Libraries imported


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

In [ ]:
# @title Load & Process Phenotypic Data
# Cell 2: Load and Process Phenotypic Data
csv1_path = "/content/drive/MyDrive/Indigo_Research/Peking_1_TestRelease_phenotypic.csv"
csv2_path = "/content/drive/MyDrive/Indigo_Research/Peking_1_phenotypic.csv"

# Load both CSVs
pheno_df1 = pd.read_csv(csv1_path)
pheno_df2 = pd.read_csv(csv2_path)

# Rename 'ID' to 'ScanDir ID' in the first dataframe
pheno_df1 = pheno_df1.rename(columns={'ID': 'ScanDir ID'})

# Merge them
pheno_df = pd.concat([pheno_df1, pheno_df2], ignore_index=True)

# Clean subject IDs
pheno_df = pheno_df.dropna(subset=['ScanDir ID'])
pheno_df['ScanDir ID'] = pheno_df['ScanDir ID'].astype(str).str.replace(r'\.0$', '', regex=True)

# For REGRESSION: Use 'ADHD Index' as target
target_column = 'ADHD Index'  # Continuous score for regression
print(f"Target variable: {target_column}")

# Replace -999 with NaN (missing values indicator in ADHD-200)
pheno_df = pheno_df.replace(-999, np.nan)

# Check missing values
print(f"\nMissing values in {target_column}: {pheno_df[target_column].isna().sum()}")
print(f"Total subjects: {len(pheno_df)}")

# Remove subjects with missing ADHD Index
pheno_df = pheno_df.dropna(subset=[target_column])
print(f"Subjects with valid ADHD Index: {len(pheno_df)}")

# Also remove subjects with missing diagnosis (for optional stratification)
pheno_df = pheno_df.dropna(subset=['DX'])
print(f"Subjects with valid diagnosis: {len(pheno_df)}")

# Check ADHD Index distribution
print(f"\nADHD Index Statistics:")
print(f"  Mean: {pheno_df[target_column].mean():.2f}")
print(f"  Std: {pheno_df[target_column].std():.2f}")
print(f"  Min: {pheno_df[target_column].min():.2f}")
print(f"  Max: {pheno_df[target_column].max():.2f}")
print(f"  Range: {pheno_df[target_column].max() - pheno_df[target_column].min():.2f}")

# Check diagnosis distribution (for stratification if desired)
print(f"\nDiagnosis distribution (DX):")
dx_counts = pheno_df['DX'].value_counts().sort_index()
for dx, count in dx_counts.items():
    dx_name = ['TDC', 'ADHD-C', 'ADHD-HI', 'ADHD-I'][int(dx)]
    print(f"  {dx_name} (Class {int(dx)}): {count} subjects")

print(f"\nSample Subject IDs: {pheno_df['ScanDir ID'].head().tolist()}")

Target variable: ADHD Index

Missing values in ADHD Index: 13
Total subjects: 136
Subjects with valid ADHD Index: 123
Subjects with valid diagnosis: 123

ADHD Index Statistics:
  Mean: 37.11
  Std: 11.70
  Min: 19.00
  Max: 68.00
  Range: 49.00

Diagnosis distribution (DX):
  TDC (Class 0): 78 subjects
  ADHD-C (Class 1): 15 subjects
  ADHD-HI (Class 2): 1 subjects
  ADHD-I (Class 3): 29 subjects

Sample Subject IDs: ['1038415', '1201251', '1245758', '1253411', '1419103']


In [ ]:
# @title Feature Extraction - Force Run (No existence check)
# Cell 3: Feature Extraction (always runs, overwrites existing)

# Define paths
fmri_dir = "/content/drive/MyDrive/Indigo_Research/fmri_preprocessed"
output_dir = "/content/drive/MyDrive/Indigo_Research/processed_features"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Path for saved features (will be overwritten)
features_save_path = os.path.join(output_dir, 'extracted_features.pkl')

print("🔄 Starting feature extraction (will overwrite any existing file)...")

# Make sure directory ends with slash
if not fmri_dir.endswith('/'):
    fmri_dir = fmri_dir + '/'

# Import required nilearn modules
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

# Atlas and masker
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100)
masker = NiftiLabelsMasker(labels_img=atlas.maps,
                          standardize=True,
                          memory='nilearn_cache',
                          verbose=1)

# Connectivity measure
connectivity_measure = ConnectivityMeasure(kind='correlation')

# Lists to store features and labels
connectomes = []
labels = []
subject_ids = []
failed_subjects = []

print(f"Processing {len(pheno_df)} subjects...")

for idx, row in pheno_df.iterrows():
    subject_id = str(row['ScanDir ID']).strip()

    # Construct file path
    fmri_path = f"{fmri_dir}sub-{subject_id}_task-rest_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"

    # Check if file exists
    if not os.path.exists(fmri_path):
        # Try alternative pattern
        fmri_path_alt = f"{fmri_dir}{subject_id}_rest.nii.gz"
        if os.path.exists(fmri_path_alt):
            fmri_path = fmri_path_alt
        else:
            print(f"✗ File not found for subject {subject_id}")
            failed_subjects.append(subject_id)
            continue

    try:
        # Extract timeseries and compute connectome
        time_series = masker.fit_transform(fmri_path)
        connectome = connectivity_measure.fit_transform([time_series])[0]

        # Get upper triangle
        mask = np.triu(np.ones(connectome.shape), k=1).astype(bool)
        features = connectome[mask]

        connectomes.append(features)
        labels.append(row[target_column])
        subject_ids.append(subject_id)

        print(f"✓ Processed subject {subject_id} (ADHD Index: {row[target_column]:.1f})")

    except Exception as e:
        print(f"✗ Failed for subject {subject_id}: {str(e)[:100]}")
        failed_subjects.append(subject_id)
        continue

# Convert to arrays
X = np.array(connectomes)
y = np.array(labels)

print(f"\n✅ Feature extraction complete!")
print(f"Successfully processed: {len(connectomes)} subjects")
print(f"Failed: {len(failed_subjects)} subjects")
print(f"Final dataset shape: {X.shape}")
print(f"ADHD Index range: [{y.min():.1f}, {y.max():.1f}]")

# Save the extracted features (overwrite)
print(f"\n💾 Saving features to {features_save_path}...")
saved_data = {
    'X': X,
    'y': y,
    'subject_ids': subject_ids,
    'failed_subjects': failed_subjects,
    'pheno_df_used': pheno_df[['ScanDir ID', target_column, 'DX']].to_dict(),
    'feature_shape': X.shape,
    'timestamp': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(features_save_path, 'wb') as f:
    pickle.dump(saved_data, f)

# Also save as numpy files for easy loading
np.save(os.path.join(output_dir, 'X_features.npy'), X)
np.save(os.path.join(output_dir, 'y_labels.npy'), y)
np.save(os.path.join(output_dir, 'subject_ids.npy'), np.array(subject_ids))

print(f"\n📁 Files saved in: {output_dir}")
print(f"  - extracted_features.pkl")
print(f"  - X_features.npy")
print(f"  - y_labels.npy")
print(f"  - subject_ids.npy")

🔄 Starting feature extraction (will overwrite any existing file)...


[fetch_atlas_schaefer_2018] Added README.md to /root/nilearn_data

[fetch_atlas_schaefer_2018] Dataset created in /root/nilearn_data/schaefer_2018

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order.txt ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

...

[NiftiLabelsMasker.wrapped] Loading data from 
'/content/drive/MyDrive/Indigo_Research/fmri_preprocessed/sub-9887336_task-rest_space-MNI152NLin2009cAsym_desc-prep
roc_bold.nii.gz'

[NiftiLabelsMasker.wrapped] Extracting region signals

[NiftiLabelsMasker.wrapped] Cleaning extracted signals

_______________________________________________filter_and_extract - 4.9s, 0.1min
✓ Processed subject 9887336 (ADHD Index: 24.0)

✅ Feature extraction complete!
Successfully processed: 123 subjects
Failed: 0 subjects
Final dataset shape: (123, 4950)
ADHD Index range: [19.0, 68.0]

💾 Saving features to /content/drive/MyDrive/Indigo_Research/processed_features/extracted_features.pkl...

📁 Files saved in: /content/drive/MyDrive/Indigo_Research/processed_features
  - extracted_features.pkl
  - X_features.npy
  - y_labels.npy
  - subject_ids.npy


/tmp/ipykernel_22861/492125571.py:63: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  connectome = connectivity_measure.fit_transform([time_series])[0]


In [ ]:
# @title Feature Extraction - Save both flattened and full matrices
# Cell 3: Feature Extraction (overwrites old files, now saves full matrices)

fmri_dir = "/content/drive/MyDrive/Indigo_Research/fmri_preprocessed"
output_dir = "/content/drive/MyDrive/Indigo_Research/processed_features"
os.makedirs(output_dir, exist_ok=True)

if not fmri_dir.endswith('/'):
    fmri_dir = fmri_dir + '/'

from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100)
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True, memory='nilearn_cache', verbose=1)
connectivity_measure = ConnectivityMeasure(kind='correlation')

# NEW: separate lists
flattened_features = []   # will be shape (n_subjects, 4950)
full_connectomes = []     # will be shape (n_subjects, 100, 100)
labels = []
subject_ids = []
failed_subjects = []

print(f"Processing {len(pheno_df)} subjects...")

for idx, row in pheno_df.iterrows():
    subject_id = str(row['ScanDir ID']).strip()
    fmri_path = f"{fmri_dir}sub-{subject_id}_task-rest_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    if not os.path.exists(fmri_path):
        fmri_path_alt = f"{fmri_dir}{subject_id}_rest.nii.gz"
        if not os.path.exists(fmri_path_alt):
            print(f"✗ File not found for subject {subject_id}")
            failed_subjects.append(subject_id)
            continue
        fmri_path = fmri_path_alt

    try:
        time_series = masker.fit_transform(fmri_path)
        connectome = connectivity_measure.fit_transform([time_series])[0]   # 2D matrix
        full_connectomes.append(connectome)                                 # store full matrix

        # flattened version for original model
        mask = np.triu(np.ones(connectome.shape), k=1).astype(bool)
        features = connectome[mask]
        flattened_features.append(features)

        labels.append(row[target_column])
        subject_ids.append(subject_id)
        print(f"✓ Processed subject {subject_id} (ADHD Index: {row[target_column]:.1f})")
    except Exception as e:
        print(f"✗ Failed for subject {subject_id}: {str(e)[:100]}")
        failed_subjects.append(subject_id)

# Convert to arrays
X_flattened = np.array(flattened_features)   # for your regression/classification models
X_full = np.array(full_connectomes)          # for graph metrics / NBS
y = np.array(labels)

# Save both
np.save(os.path.join(output_dir, 'X_features.npy'), X_flattened)          # your original flattened features
np.save(os.path.join(output_dir, 'X_full_connectomes.npy'), X_full)       # NEW: 100x100 matrices
np.save(os.path.join(output_dir, 'y_labels.npy'), y)
np.save(os.path.join(output_dir, 'subject_ids.npy'), np.array(subject_ids))

# Also save the combined pickle (optional)
with open(os.path.join(output_dir, 'extracted_features.pkl'), 'wb') as f:
    pickle.dump({
        'X': X_flattened,
        'X_full': X_full,
        'y': y,
        'subject_ids': subject_ids,
        'failed_subjects': failed_subjects,
        'timestamp': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    }, f)

print(f"\n✅ Saved flattened features: {X_flattened.shape}")
print(f"✅ Saved full connectomes: {X_full.shape}")
print(f"✅ Labels: {y.shape}")

[fetch_atlas_schaefer_2018] Dataset found in /root/nilearn_data/schaefer_2018

Processing 123 subjects...


[NiftiLabelsMasker.wrapped] Loading regions from 
'/root/nilearn_data/schaefer_2018/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_1mm.nii.gz'

[NiftiLabelsMasker.wrapped] Resampling regions

[NiftiLabelsMasker.wrapped] Finished fit

...

[NiftiLabelsMasker.wrapped] Resampling regions

[NiftiLabelsMasker.wrapped] Finished fit

/tmp/ipykernel_22861/3525397730.py:40: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  time_series = masker.fit_transform(fmri_path)


✓ Processed subject 9887336 (ADHD Index: 24.0)

✅ Saved flattened features: (123, 4950)
✅ Saved full connectomes: (123, 100, 100)
✅ Labels: (123,)


/tmp/ipykernel_22861/3525397730.py:41: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  connectome = connectivity_measure.fit_transform([time_series])[0]   # 2D matrix


In [ ]:
# ============================================================
# Cell [EXPERIMENTAL] Graph metrics from full connectomes
# ============================================================
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
import numpy as np
import os

def compute_graph_metrics(corr_matrix):
    """
    corr_matrix : 2D numpy array (e.g., 100x100)
    Returns [modularity, global_efficiency, characteristic_path_length]
    """
    # Build graph from correlation matrix (keep positive edges only)
    G = nx.from_numpy_array(corr_matrix)
    edges_to_remove = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] <= 0]
    G.remove_edges_from(edges_to_remove)

    if G.number_of_nodes() == 0:
        return np.array([0.0, 0.0, 0.0])

    # Modularity (using greedy communities)
    try:
        communities = list(greedy_modularity_communities(G))
        modularity = nx.community.modularity(G, communities)
    except:
        modularity = 0.0

    # Global efficiency
    efficiency = nx.global_efficiency(G)

    # Characteristic path length (largest connected component)
    if nx.is_connected(G):
        path_length = nx.average_shortest_path_length(G)
    else:
        largest_cc = max(nx.connected_components(G), key=len)
        subG = G.subgraph(largest_cc)
        path_length = nx.average_shortest_path_length(subG) if subG.number_of_nodes() > 1 else 0.0

    return np.array([modularity, efficiency, path_length])

# Load the full connectomes (100x100 matrices) saved during extraction
processed_dir = "/content/drive/MyDrive/Indigo_Research/processed_features"
X_full = np.load(os.path.join(processed_dir, 'X_full_connectomes.npy'))
print(f"Loaded full connectomes: {X_full.shape}")   # should be (n_subjects, 100, 100)

# Compute graph metrics for every subject
graph_features_list = []
for conn in X_full:   # now conn is 100x100, not 1D
    graph_features_list.append(compute_graph_metrics(conn))

X_graph = np.array(graph_features_list)   # shape = (n_subjects, 3)
print(f"Graph feature matrix shape: {X_graph.shape}")

# Save graph features for later use
np.save(os.path.join(processed_dir, 'X_graph_features.npy'), X_graph)
print("✅ Graph features saved to X_graph_features.npy")

Loaded full connectomes: (123, 100, 100)
Graph feature matrix shape: (123, 3)
✅ Graph features saved to X_graph_features.npy
